In [1]:
from pathlib import Path
import os
from os import PathLike
from typing import Optional, Sequence, Union
from fastcore.basics import patch

from trouver.obsidian.vault import VaultNote, note_name_from_path
from trouver.obsidian.links import ObsidianLink, LinkType, replace_links_in_text

In [2]:
import shutil
import subprocess
import warnings
import tempfile
from unittest import mock

from fastcore.test import *
from nbdev.showdoc import show_doc

from trouver.helper.tests import _test_directory

## Creating/deleting/moving the note

If a `VaultNote` object represents a non-existent file, then the file can be created with empty content. The cache is updated with this single new entry, but the rest of the cache remains the same.

If the file exists, then a `FileExistsError` is raised.

If the specified directory for the file does not exist, then a `FileNotFoundError` is raised.

In [3]:
#| export obsidian.vault
@patch
def create(self: VaultNote):
    # TODO: consider using _check_name_exists_and_unique_in_vault_cache method.
    r"""Create the note if it does not exist.
    
    The directory of the file needs to be created separately
    beforehand.

    If the file exists, then a FileExistsError is raised and
    the file modification time is not changed.
    
    **Raises**

    - FileExistsError
        - If the file already exists.
    - FileNotFoundError
        - If the directory of the file does not already exist.
    """
    Path(self.path()).touch(exist_ok=False)
    self.__class__._add_single_entry_to_cache(
        self.vault, self.rel_path)

In [4]:
show_doc(VaultNote.create)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/main/trouver/obsidian/vault.py#L628){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.create

```python

def create(
    
):


```

*Create the note if it does not exist.*

The directory of the file needs to be created separately
beforehand.

If the file exists, then a FileExistsError is raised and
the file modification time is not changed.

**Raises**

- FileExistsError
    - If the file already exists.
- FileNotFoundError
    - If the directory of the file does not already exist.

In [5]:
#| export obsidian.vault
@patch
def delete(self: VaultNote) -> None:
    r"""Delete the note if it exists.
    
    This updates the cache if necessary
    """
    if not self.exists(update_cache=True):
        return
    os.remove(self.path())
    self.__class__._remove_single_entry_from_cache(
        self.vault, self.rel_path)

In [6]:
show_doc(VaultNote.delete)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/main/trouver/obsidian/vault.py#L651){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.delete

```python

def delete(
    
)->None:


```

*Delete the note if it exists.*

This updates the cache if necessary

In [7]:
#| export obsidian.vault
@patch
def move_to(self: VaultNote,
            rel_path: PathLike # The path in which to rename the path to `self` as, relative to `self.vault`.
            ) -> None:
    r"""Move/rename the note to the specified location in the vault,
    assuming that it exists.

    Nothing is done if the note does not exist.
    """
    if not self.exists():
        return
    os.rename(self.path(), Path(self.vault) / rel_path)
    self.__class__._remove_single_entry_from_cache(
        self.vault, Path(self.rel_path))
    self.__class__._add_single_entry_to_cache(
        self.vault, Path(rel_path))
    self.rel_path = str(rel_path)
    self.name = note_name_from_path(self.rel_path)
    
@patch
def move_to_folder(self: VaultNote,
                    rel_dir: PathLike # The path of the directory in which to move `self` to, relative to `self.vault`.
                    ) -> None:
    # TODO: consider using _check_name_exists_and_unique_in_vault_cache method.
    r"""Move the note to the specified folder in the vault, assuming that
    it exists.
    """
    self.move_to(Path(rel_dir) / f'{self.name}.md')

In [8]:
show_doc(VaultNote.move_to)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/main/trouver/obsidian/vault.py#L664){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.move_to

```python

def move_to(
    rel_path:PathLike, # The path in which to rename the path to `self` as, relative to `self.vault`.
)->None:


```

*Move/rename the note to the specified location in the vault,*
assuming that it exists.

Nothing is done if the note does not exist.

In [9]:
show_doc(VaultNote.move_to_folder)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/main/trouver/obsidian/vault.py#L683){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.move_to_folder

```python

def move_to_folder(
    rel_dir:PathLike, # The path of the directory in which to move `self` to, relative to `self.vault`.
)->None:


```

*Move the note to the specified folder in the vault, assuming that*
it exists.

### Fast candidate discovery for link replacement

`VaultNote.rename` must rewrite links in every note that references the
renamed note, and a reference can sit anywhere in the vault. Scanning all
~160k notes of a large vault file by file is slow (tens of seconds even at
the raw-I/O floor); ripgrep does the same literal substring search in
seconds (parallel walk, SIMD search). Since *every* link style to
`old_name` contains `old_name` as a literal substring -- the very
invariant the per-note pre-check exploits -- a files-with-matches pass
over the raw text is a sound superset of the truly referencing notes: the
regex passes still make every replacement decision.

The two helpers below are optional accelerators: when ripgrep is absent,
`rename` falls back to scanning every cached note, with identical
results.

In [ ]:
#| export obsidian.vault
def _ripgrep_path() -> Optional[Path]: # Path to the ripgrep executable, or `None` if not found.
    r"""Return the path to the ripgrep (`rg`) executable, or `None` if it
    is not available.

    Resolution order: the `TROUVER_RG_PATH` environment variable (for
    machines where ripgrep is installed but not on `PATH`), then a `rg`
    lookup on `PATH`. Re-resolved on every call so installing ripgrep
    (or setting `TROUVER_RG_PATH`) takes effect without a restart.
    """
    env_path = os.environ.get('TROUVER_RG_PATH')
    if env_path and os.path.isfile(env_path):
        return Path(env_path)
    found = shutil.which('rg')
    return Path(found) if found is not None else None

In [ ]:
#| export obsidian.vault
# Vault size at or above which falling back to a full scan is slow
# enough (tens of seconds) that the missing-ripgrep warning is worth
# its noise; smaller vaults fall back silently.
_RIPGREP_FALLBACK_WARN_SIZE = 2000

def _note_paths_containing_literal(
        needle: str, # The literal text to search for.
        vault: PathLike # The vault in which to search.
        ) -> Optional[list[Path]]: # Vault-relative paths of the `.md` files containing `needle`, or `None` if ripgrep is unavailable or the search failed.
    r"""Return the vault-relative paths of Markdown notes whose text
    contains `needle` as a literal substring, using ripgrep.

    Returns `None` when ripgrep is unavailable or fails, so callers can
    fall back to scanning every note themselves. `--hidden --no-ignore`
    keeps the searched set a superset of `VaultNote.cache`, which an
    unrestricted `os.walk` builds (gitignored and hidden notes included);
    `-l --fixed-strings` is a literal files-with-matches pass; NUL-separated
    output (`-0`) keeps exotic file names parseable.
    """
    rg = _ripgrep_path()
    if rg is None:
        return None
    cmd = [str(rg), '--hidden', '--no-ignore', '--no-messages', '-0',
           '-l', '--fixed-strings', '-g', '*.md', '--', needle, '.']
    try:
        result = subprocess.run(cmd, capture_output=True, cwd=str(vault))
    except OSError:
        return None
    if result.returncode not in (0, 1): # 0/1 mean hits/no-hits; others failed.
        return None
    output = result.stdout.decode('utf-8', errors='replace')
    return [Path(line) for line in output.split('\0') if line]

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    fake_rg = Path(temp_dir) / 'rg.exe'
    fake_rg.touch()
    # `TROUVER_RG_PATH` wins when it points at a real file.
    with mock.patch.dict(os.environ, {'TROUVER_RG_PATH': str(fake_rg)}):
        test_eq(_ripgrep_path(), fake_rg)
    # An unusable `TROUVER_RG_PATH` falls through to the PATH lookup.
    env_no_var = {k: v for k, v in os.environ.items() if k != 'TROUVER_RG_PATH'}
    with mock.patch.dict(os.environ, env_no_var, clear=True), \
         mock.patch('shutil.which', return_value=None):
        test_eq(_ripgrep_path(), None)
    with mock.patch.dict(os.environ, {'TROUVER_RG_PATH': str(Path(temp_dir) / 'missing.exe')}), \
         mock.patch('shutil.which', return_value='/usr/bin/rg'):
        test_eq(_ripgrep_path(), Path('/usr/bin/rg'))

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    fake_rg = Path(temp_dir) / 'rg.exe'
    fake_rg.touch()
    env_rg = {'TROUVER_RG_PATH': str(fake_rg)}
    def run_return(returncode, stdout=b''):
        return mock.Mock(return_value=mock.Mock(returncode=returncode,
                                                stdout=stdout))
    # NUL-separated hits parse to vault-relative Paths; the trailing
    # empty fragment is dropped.
    with mock.patch.dict(os.environ, env_rg), \
         mock.patch('subprocess.run', run_return(0, b'a.md\x00b c.md\x00')):
        test_eq(_note_paths_containing_literal('x', temp_dir),
                [Path('a.md'), Path('b c.md')])
    # Returncode 1 means "no matches" -- an empty list, not a failure.
    with mock.patch.dict(os.environ, env_rg), \
         mock.patch('subprocess.run', run_return(1)):
        test_eq(_note_paths_containing_literal('x', temp_dir), [])
    # Any other returncode, or an unlaunchable rg, means "fall back".
    with mock.patch.dict(os.environ, env_rg), \
         mock.patch('subprocess.run', run_return(2)):
        test_eq(_note_paths_containing_literal('x', temp_dir), None)
    with mock.patch.dict(os.environ, env_rg), \
         mock.patch('subprocess.run', side_effect=OSError):
        test_eq(_note_paths_containing_literal('x', temp_dir), None)
    # No ripgrep at all -> `None` without attempting a search.
    env_no_var = {k: v for k, v in os.environ.items() if k != 'TROUVER_RG_PATH'}
    with mock.patch.dict(os.environ, env_no_var, clear=True), \
         mock.patch('shutil.which', return_value=None):
        test_eq(_note_paths_containing_literal('x', temp_dir), None)

In [10]:
#| export obsidian.vault
def _replace_links_all_styles(
        text: str, # The text in which to replace links.
        old_name: str, # The name that linked-to notes referred to before the rename.
        new_name: str, # The new name to point links at.
        ) -> str:
    r"""Return `text` with all wikilink- and markdown-style links to
    `old_name` retargeted to `new_name`.

    Every pattern matches on the literal `old_name`, so callers may skip
    this function whenever `old_name` is not a substring of `text`.
    """
    patterns_and_new_names = [
        (ObsidianLink(
            is_embedded=-1, file_name=old_name, anchor=-1, custom_text=-1),
         new_name),
        (ObsidianLink(
            is_embedded=-1, file_name=f'{old_name}.md', anchor=-1,
            custom_text=-1, link_type=LinkType.MARKDOWN),
         f'{new_name}.md'),
        (ObsidianLink(
            is_embedded=-1, file_name=old_name, anchor=-1,
            custom_text=-1, link_type=LinkType.MARKDOWN),
         new_name)]
    for pattern, replacement in patterns_and_new_names:
        text = replace_links_in_text(
            text, pattern, new_link_name=replacement)
    return text

def _replace_links_in_note_paths(
        vault: PathLike, # The vault.
        rel_paths: Sequence[PathLike], # Candidate notes to inspect, relative to `vault`.
        old_name: str, # The name that linked-to notes referred to before the rename.
        new_name: str # The new name to point links at.
        ) -> None:
    r"""Rewrite every wikilink- and markdown-style link to `old_name`
    into `new_name` in each of `rel_paths`.

    Notes whose text lacks `old_name` as a substring are skipped before
    the regex passes (no link pattern can match without it), so callers
    may pass any superset of the truly referencing notes.
    """
    for rel_path in rel_paths:
        other_note = VaultNote(vault=vault, rel_path=rel_path)
        if not other_note.exists():
            continue
        original_text = other_note.text()
        # Every link pattern contains `old_name` literally, so a note
        # without the substring cannot reference the renamed note and
        # needs no regex passes. (See `_replace_links_all_styles`.)
        if old_name not in original_text:
            continue
        text = _replace_links_all_styles(original_text, old_name, new_name)
        if text == original_text:
            continue
        with open(other_note.path(), 'w', encoding='utf-8') as file:
            file.write(text)

@patch
def rename(
        self: VaultNote,
        new_name: str,  # The new name to give the note('s file). Should not include the `.md` extension.`
        replace_links_in_vault: bool = True  # If `True`, then replace the links in the vault pointing to `note` to reflect the new name.
        ) -> None:

    # TODO: consider using _check_name_exists_and_unique_in_vault_cache method.
    r"""
    Rename the file underlying `self` to `new_name`. The directory that the file is in remains unchanged.

    Assumes that

    1. the name of `self` is unique among note names in `self.vault`s
    2. No pre-existing note in `vault` has `new_name` as its name. Use the `note_name_unique`
        function to check whether or not this is the case.
    3. the class' `.cache` accurately reflects the files in `vault`

    Every note in the vault must be read (references can be anywhere), but
    the regex replacement passes are skipped for notes that do not contain
    `old_name` as a substring -- no link pattern can match without it.

    When ripgrep is available (see `_ripgrep_path`), only the notes whose
    text contains `old_name` are read; otherwise every cached note is
    scanned, with a one-time warning when the vault is large enough for
    that scan to be slow (see `_RIPGREP_FALLBACK_WARN_SIZE`). Both routes
    produce identical results, since every link to `old_name` contains it
    as a substring.

    """
    old_name = self.name
    parent_dir = Path(os.path.dirname(self.rel_path))
    self.move_to(parent_dir / f'{new_name}.md')
    if not replace_links_in_vault:
        return
    candidates = _note_paths_containing_literal(old_name, self.vault)
    if candidates is None: # No ripgrep -> scan every cached note.
        candidates = [path for paths in
                      self.__class__.cache[str(self.vault)].values()
                      for path in paths]
        if len(candidates) >= _RIPGREP_FALLBACK_WARN_SIZE:
            warnings.warn(
                f'`VaultNote.rename` is scanning all {len(candidates)} '
                'notes to find links to rewrite because ripgrep (`rg`) '
                'was not found. Install it (e.g. `winget install '
                'BurntSushi.ripgrep.MSVC`, then restart running apps) or '
                'set `TROUVER_RG_PATH` for a several-fold speedup.',
                stacklevel=2)
    _replace_links_in_note_paths(
        self.vault, candidates, old_name, new_name)

In [11]:
show_doc(VaultNote.rename)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/main/trouver/obsidian/vault.py#L694){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.rename

```python

def rename(
    new_name:str, # The new name to give the note('s file). Should not include the `.md` extension.`
    replace_links_in_vault:bool=True, # If `True`, then replace the links in the vault pointing to `note` to reflect the new name.
):


```

*Rename the file underlying `self` to `new_name`. The directory that the file is in remains unchanged.*

Assumes that

1. the name of `self` is unique among note names in `self.vault`s
2. No pre-existing note in `vault` has `new_name` as its name. Use the `note_name_unique`
    function to check whether or not this is the case.
3. the class' `.cache` accurately reflects the files in `vault`

In [12]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    vault_note = VaultNote(temp_vault, rel_path='new_file.md')
    vault_note.create()
    assert vault_note.exists()
    assert vault_note.rel_path in VaultNote.cache[str(temp_vault)]['new_file']

    with ExceptionExpected(ex=FileExistsError):
        vault_note.create() 

    vault_note = VaultNote(temp_vault, rel_path='none_existent_folder/new_file.md')
    with ExceptionExpected(ex=FileNotFoundError):
        vault_note.create()

In [13]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    vault_note = VaultNote(temp_vault, name='exponential_function')
    vault_note.delete()
    assert not vault_note.exists()
    assert vault_note.rel_path not in VaultNote.cache[str(temp_vault)]['exponential_function']

If a `VaultNote` object represents an existing file, then the file can be renamed or moved.

In [14]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    VaultNote.clear_cache()
    vault_note = VaultNote(temp_vault, name='exponential_function')
    vault_note.move_to_folder('')
    assert vault_note.rel_path == 'exponential_function.md'
    assert 'exponential_function.md' in VaultNote.cache[str(temp_vault)]['exponential_function']

The `VaultNote.rename` function renames the file underlying an existing vault note.

In [15]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    vault_note = VaultNote(temp_vault, name='exponential_function')
    print(f"Old path: {vault_note.rel_path}")
    vault_note.rename('exp_func')

    expected_path = os.path.normpath('analysis/exp_func.md')
    actual_path = os.path.normpath(vault_note.rel_path)
    test_eq(actual_path, expected_path)

    # test_eq(Path(vault_note.rel_path), Path(r'analysis\exp_func.md'))
    assert vault_note.exists()
    print(f"New path: {vault_note.rel_path}")

Old path: analysis\exponential_function.md
New path: analysis\exp_func.md


By default, the `rename` method replaces links in notes in the vault 

In [16]:

with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    vault_note = VaultNote(temp_vault, name='note_1')
    other_note = VaultNote(temp_vault, name='note_2_with_links_to_note_1')
    print(f"Old path: {vault_note.rel_path}")
    print(f"`note_2_with_links_to_note_1` is a note with links to `note_1`. The following is its contents:")
    print(other_note.text())
    vault_note.rename('renamed_note_1')

    assert other_note.exists()
    print(f"\nNew path: {vault_note.rel_path}")
    print(f"After `note_1` is renamed, the following is the contents of `note_2_with_links_to_note_1`:")

    new_text_in_note_2 = other_note.text()
    print(new_text_in_note_2)
    assert '[[renamed_note_1]]' in new_text_in_note_2
    assert '[[renamed_note_1|hi]]' in new_text_in_note_2
    assert '[[renamed_note_1#blahblah|hi]]' in new_text_in_note_2
    assert '![[renamed_note_1|embedded]]' in new_text_in_note_2
    assert '[asdf](renamed_note_1)' in new_text_in_note_2
    assert '[asdf](renamed_note_1.md)' in new_text_in_note_2

Old path: topology\note_1.md
`note_2_with_links_to_note_1` is a note with links to `note_1`. The following is its contents:
[[note_1]]

[[note_1|hi]]
[[note_1#blahblah|hi]]

![[note_1|embedded]]

[asdf](note_1)

[asdf](note_1.md)

New path: topology\renamed_note_1.md
After `note_1` is renamed, the following is the contents of `note_2_with_links_to_note_1`:
[[renamed_note_1]]

[[renamed_note_1|hi]]
[[renamed_note_1#blahblah|hi]]

![[renamed_note_1|embedded]]

[asdf](renamed_note_1)

[asdf](renamed_note_1.md)


In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    # A bystander whose text contains only NEAR MISSES of the old name
    # (no literal substring 'renameme') must be skipped by the substring
    # pre-check and left byte-identical.
    VaultNote(temp_vault, rel_path='renameme.md').create()
    bystander = VaultNote(temp_vault, rel_path='bystander.md')
    bystander.create()
    near_miss_text = (
        'Mentions [[other_note]], renamex, "rename me", and '
        '[[target_x|a renamexe alias]].\n')
    with open(bystander.path(), 'w', encoding='utf-8') as file:
        file.write(near_miss_text)

    VaultNote(temp_vault, name='renameme').rename('renamednow')

    with open(bystander.path(), encoding='utf-8') as file:
        test_eq(file.read(), near_miss_text)

    # Referrers that DO contain the old name are still rewritten in every
    # link style (alias wikilink, embed, markdown `.md` link). Kept one
    # style per line for clarity; combined styles on a single line only
    # became reliable with the 2026-10-20 `to_regex` bracket-exclusion fix
    # (see the characterization cell at the end of 03_obsidian_50.links).
    referrer = VaultNote(temp_vault, rel_path='referrer.md')
    referrer.create()
    with open(referrer.path(), 'w', encoding='utf-8') as file:
        file.write('[[renamednow|alias]]\n![[renamednow]]\n[x](renamednow.md)\n')
    # A link to a LONGER name that merely starts with the old name passes
    # the substring pre-check; the regexes must still leave it alone.
    longer = VaultNote(temp_vault, rel_path='longername.md')
    longer.create()
    longer_text = '[[renamednow_but_different]] links elsewhere.\n'
    with open(longer.path(), 'w', encoding='utf-8') as file:
        file.write(longer_text)

    VaultNote(temp_vault, name='renamednow').rename('renamedagain')

    test_eq(longer.text(), longer_text)
    new_text = referrer.text()
    assert '[[renamedagain|alias]]' in new_text
    assert '![[renamedagain]]' in new_text
    assert '[x](renamedagain.md)' in new_text

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    # Referrers the plain scan finds but a default ripgrep run would SKIP:
    # one inside a gitignored directory, one inside a hidden directory.
    (temp_vault / 'ignored_dir').mkdir()
    with open(temp_vault / '.gitignore', 'w', encoding='utf-8') as file:
        file.write('ignored_dir/\n')
    ignored = VaultNote(temp_vault, rel_path=Path('ignored_dir') / 'ref.md')
    ignored.create()
    ignored.write('Links [[target]] here.\n')
    (temp_vault / '.hidden_dir').mkdir()
    hidden = VaultNote(temp_vault, rel_path=Path('.hidden_dir') / 'ref.md')
    hidden.create()
    hidden.write('Also links [[target]].\n')
    target = VaultNote(temp_vault, rel_path='target.md')
    target.create()

    # Candidate discovery (when ripgrep is available) must cover at least
    # every note containing the literal name -- including both referrers.
    candidates = _note_paths_containing_literal('target', temp_vault)
    if candidates is not None:
        assert Path('ignored_dir') / 'ref.md' in candidates
        assert Path('.hidden_dir') / 'ref.md' in candidates
        assert Path('bystander_absent.md') not in candidates

    VaultNote(temp_vault, name='target').rename('newtarget')

    assert '[[newtarget]]' in ignored.text()
    assert '[[newtarget]]' in hidden.text()

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    target = VaultNote(temp_vault, rel_path='target.md')
    target.create()
    env_no_var = {k: v for k, v in os.environ.items() if k != 'TROUVER_RG_PATH'}

    def rename_wo_rg(note, new_name):
        # Force the fallback route regardless of whether rg is installed.
        with mock.patch.dict(os.environ, env_no_var, clear=True), \
             mock.patch('shutil.which', return_value=None), \
             warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            note.rename(new_name)
        return caught

    # Small vault -> fallback is instant -> no warning.
    caught = rename_wo_rg(VaultNote(temp_vault, name='target'), 'target2')
    test_eq(len(caught), 0)

    # Pad the cache past the threshold (paths need not exist -- the scan
    # skips them) -> the fallback warns about missing ripgrep.
    VaultNote.cache[str(temp_vault)].update(
        {f'pad_{i}': [f'pad_{i}.md'] for i in range(_RIPGREP_FALLBACK_WARN_SIZE)})
    caught = rename_wo_rg(VaultNote(temp_vault, name='target2'), 'target3')
    test_eq(len(caught), 1)
    assert issubclass(caught[0].category, UserWarning)
    assert 'ripgrep' in str(caught[0].message)

### Getting a unique note name

It is troublesome to create a note with a non-unique name. The `unique_name` method of the VaultNote class takes a tentative name for a note to be created and adds a number to the name so that the note name will be unique in the vault. 

In [17]:
#| export obsidian.vault

@patch(cls_method=True)
def unique_name(
        cls: VaultNote,
        name: str, # The base name for the note.
        vault: PathLike, # The vault
        other_unavailable_names: Optional[Sequence[str]] = None # Other names that should be excluded when generating the unique name
        ) -> str: # A str obtained by appending `_{some number}` to the end of `name`.
    r"""A class method to return a name for a note that is unique in
    the vault based on a specified name.
    """
    if not str(vault) in VaultNote.cache:
        cls.update_cache(vault)
    unavailable_names = set(VaultNote.cache[str(vault)])
    if other_unavailable_names is None:
        other_unavailable_names = []
    unavailable_names.update(other_unavailable_names)
    if not name in unavailable_names:
    # if not name in VaultNote.cache[str(vault)]:
        return name
    i = 1
    # while f'{name}_{i}' in VaultNote.cache[str(vault)]:
    while f'{name}_{i}' in unavailable_names:
        i += 1
    return f'{name}_{i}'

In [18]:
show_doc(VaultNote.unique_name)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/main/trouver/obsidian/vault.py#L739){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.unique_name

```python

def unique_name(
    name:str, # The base name for the note.
    vault:PathLike, # The vault
    other_unavailable_names:Optional=None, # Other names that should be excluded when generating the unique name
)->str: # A str obtained by appending `_{some number}` to the end of `name`.


```

*A class method to return a name for a note that is unique in*
the vault based on a specified name.

In [19]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    # There is a note named category in the test vault.
    sample_name = VaultNote.unique_name('category', temp_vault)  
    assert not VaultNote(temp_vault, name=sample_name).exists()

    sample_name = VaultNote.unique_name('non_existent_note_name', temp_vault)
    assert not VaultNote(temp_vault, name=sample_name).exists()
    assert sample_name == 'non_existent_note_name'

    sample_name = VaultNote.unique_name('category', temp_vault, other_unavailable_names=['category_1']) 
    assert not VaultNote(temp_vault, name=sample_name).exists()
    assert sample_name not in ['category', 'category_1']

In [20]:
#| hide
# This import 
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    # Test _add_single_entry_to_cache
    VaultNote.clear_cache()
    VaultNote._add_single_entry_to_cache(temp_vault, rel_path='directory/file.md')
    assert len(VaultNote.cache) == 1
    assert len(VaultNote.cache[str(temp_vault)]) == 1
    assert len(VaultNote.cache[str(temp_vault)]['file']) == 1

    VaultNote._remove_single_entry_from_cache(temp_vault, rel_path=Path('directory/file.md'))
    assert len(VaultNote.cache) == 1
    assert len(VaultNote.cache[str(temp_vault)]) == 1
    assert len(VaultNote.cache[str(temp_vault)]['file']) == 0

    # 1. Patch the utility function where it is actually used by VaultNote
    with mock.patch('trouver.obsidian.vault.all_paths_to_notes_in_vault') as mock_method:
        VaultNote.update_cache(temp_vault)
        mock_method.assert_called()

    # 2. Patch the class methods on the imported class directly
    # It is often safer/easier to use mock.patch.object for imported classes
    with (mock.patch.object(VaultNote, '_add_single_entry_to_cache') as mock_add_cache,
        mock.patch.object(VaultNote, '_remove_single_entry_from_cache') as mock_remove_cache):
        
        # Identified by `rel_path` rather than by name: the mocked
        # `update_cache` above marked the vault fresh, so a name lookup
        # would (correctly) not re-walk and could not identify the note.
        vault_note = VaultNote(
            temp_vault, rel_path=Path('analysis') / 'exponential_function.md')
        vault_note.move_to_folder('')
        
        mock_remove_cache.assert_called_with(temp_vault, Path('analysis') / Path('exponential_function.md'))
        mock_add_cache.assert_called_with(temp_vault, Path('exponential_function.md'))

In [21]:
## Copying files in an `Obsidian.md` vault to and from a subvault

In [22]:
# # TODO: use these methods during vault construction for a reference
# def copy_vault_file_into_subvault(
#         vault: PathLike, # The Path to the vault from which to copy the files.
#         subvaults: Union[PathLike, list[PathLike]], # The Paths to the subvaults to which to copy the files.
#         files: Union[PathLike, list[PathLike]], # The Path to the files, relative to `vault` to copy.
#         replace: bool = True, # If `True`, replace existing files in `subvaults` if necessary. Defaults to `True`
#         backup: bool = True, # If `True` and if `replace=True`, create a backup for any replaced files in a subvault in a folder named `.back` in the root directory of the subvault.
#         ) -> None: 
#     """Copy the specified files in `vault` into subvaults.

#     The files are copied within the subvaults to the same relative paths as
#     they are found in `vault`.

#     Here, "files" include directories. If a directory is copied, then all
#     files and subdirectories of that directory are also copied.

#     **Parameters**
#     - vault - PathLike
#         - The path to the Obsidian vault from which to copy files from.
#     - subvaults - PathLike or list[PathLike]
#         - The paths to the subvaults to which to copy the files.
#     - files - PathLike or list[PathLike]
#         - The files to copy.

#     **Raises**
#     - FileExistsError
#         - If `replace` is `False` and some subvault already has a file at
#           the path in which a file-copy is attempted. In this case, no
#           files are copied.
#     - FileNotFoundError
#         - If a path specified in `files` does not exist in `vault`. In this
#           case, no files are copied.

#     """
#     vault = Path(vault)
#     if isinstance(subvaults, PathLike):
#         subvaults = [subvaults]
#     if isinstance(files, PathLike):

#         files = [files]


#     # TODO: Implement this as a private function 
#     for file in files:
#         if not os.path.exists(vault / file):
#             raise FileNotFoundError(
#                 f"Attempted to copy files/folders from a vault into subvaults"
#                 f", but there is at least one non-existent files"
#                 f". No files have been copied."
#                 f"vault: {vault}"
#                 f"file: {file}")

#     if not replace: 
#         for subvault, file in itertools.product(subvaults, files):
#             if os.path.exists(subvault / file):
#                 raise FileExistsError(
#                     f"Attempted to copy files/folders from a vault into subvaults"
#                     f", but at least one subvault already has a file that is"
#                     f" supposed to be copied from the vault"
#                     f". No files have been copied."
#                     f"subvault: {subvault}"
#                     f"file: {file}")
    
#     # TODO: copy and backup files

#     return

In [23]:
#| export obsidian.vault
def test_function():
    return None